# Single-Race Prototype

**Goal**: For one race (2023 Japan GP), build one feature row per driver using only the first half of laps, attach podium labels, and verify the output.

Reference: `misc/plan_podium_prediction.md`

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os

import kyrios.data
import kyrios.features
import kyrios.model

import importlib
importlib.reload(kyrios.data)
importlib.reload(kyrios.features)
importlib.reload(kyrios.model)

from kyrios.data import load_session
from kyrios.features import build_features
from kyrios.model import prepare, train, evaluate, feature_importance

/Users/chethanjujjavarapu/Desktop/GitHub/kyrios/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
quali = load_session(2023, 'Japan', 'Q')
race  = load_session(2023, 'Japan', 'R')

df = build_features(quali, race)
df.head()

core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '11', '55', '44', '63', '22', '14', '40', '10', '23', '31', '20', '77', '18', '27', '24', '2']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached

,Abbreviation,TeamName,GridPosition,GapToPole,InQ3,Circuit,Year,ChampPoints,ChampPosition,ChampWins,Podium
0,VER,Red Bull Racing,1.0,0.000,1,Suzuka,2023,374.0,1,12,1
1,PIA,McLaren,2.0,0.581,1,Suzuka,2023,42.0,11,0,1
2,NOR,McLaren,3.0,0.616,1,Suzuka,2023,97.0,8,0,1
3,LEC,Ferrari,4.0,0.665,1,Suzuka,2023,123.0,6,0,0
4,PER,Red Bull Racing,5.0,0.773,1,Suzuka,2023,223.0,2,2,0


In [3]:
X, y, groups = prepare(df)
print(f"Features shape: {X.shape}")
print(f"Podium distribution:\n{y.value_counts()}")
X.head()

Features shape: (20, 8)
Podium distribution:
Podium
0    17
1     3
Name: count, dtype: int64


,GridPosition,GapToPole,InQ3,Circuit,Year,ChampPoints,ChampPosition,ChampWins
0,1.0,0.000,1,0,2023,374.0,1,12
1,2.0,0.581,1,0,2023,42.0,11,0
2,3.0,0.616,1,0,2023,97.0,8,0
3,4.0,0.665,1,0,2023,123.0,6,0
4,5.0,0.773,1,0,2023,223.0,2,2


## Multi-Race Dataset
Load multiple races to have enough data for training and evaluation.

In [4]:
import fastf1

schedule_2023 = fastf1.get_event_schedule(2023)
# Only conventional races (skip testing/pre-season)
races_2023 = schedule_2023[schedule_2023['EventFormat'] != 'testing']

frames = []
for _, event in races_2023.iterrows():
    round_num = event['RoundNumber']
    name = event['EventName']
    try:
        q = load_session(2023, round_num, 'Q')
        r = load_session(2023, round_num, 'R')
        row = build_features(q, r)
        frames.append(row)
        print(f"  Round {round_num}: {name} — {len(row)} drivers")
    except Exception as e:
        print(f"  Round {round_num}: {name} — SKIPPED ({e})")

full_df = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows: {len(full_df)}, Podiums: {full_df['Podium'].sum()}")

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached d

  Round 1: Bahrain Grand Prix — 20 drivers


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['11', '16', '14', '63', '55', '18', '31', '44', '81', '10', '27', '24', '20', '77', '1', '22', '23', '21', '4', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for la

  Round 2: Saudi Arabian Grand Prix — 20 drivers


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '63', '44', '14', '55', '18', '16', '23', '10', '27', '31', '22', '4', '20', '21', '81', '24', '2', '77', '11']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_c

...

req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '1', '63', '10', '23', '2', '77', '20', '14', '44', '11', '27', '18', '3', '4', '31', '24', '81', '22']
core           INFO 	Loading data for Las Vegas Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_cou

  Round 21: Las Vegas Grand Prix — 20 drivers


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to

  Round 22: Abu Dhabi Grand Prix — 20 drivers

Total rows: 440, Podiums: 66


/var/folders/hd/jwq7dcln0d7d4w27ps134_3c0000gn/T/ipykernel_54932/1572859771.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_df = pd.concat(frames, ignore_index=True)


## Train & Evaluate

In [5]:
results = evaluate(full_df)
print(f"Mean AUC: {results['mean_auc']:.3f}")
print(f"Fold AUCs: {[round(a, 3) for a in results['fold_aucs']]}")

Mean AUC: 0.891
Fold AUCs: [np.float64(0.915), np.float64(0.934), np.float64(0.858), np.float64(0.824), np.float64(0.924)]


In [6]:
model = train(full_df)
fi = feature_importance(model, full_df)
fi

,feature,importance
0,GridPosition,0.397467
1,ChampWins,0.160815
2,ChampPosition,0.128002
3,ChampPoints,0.113677
4,GapToPole,0.083744
5,Circuit,0.062088
6,InQ3,0.054207
7,Year,0.000000
